# Milestone 2 — Data Cleaning: Corpus-Indonesia

Sumber: [Lyon28/Corpus-Indonesia](https://huggingface.co/datasets/Lyon28/Corpus-Indonesia)

Input: `data/raw/corpus-indonesia/data/*.parquet`  
Output: `data/processed/corpus_indonesia_cleaned.parquet`  
Laporan: `output/tables/data_quality_report.json`  
Ringkasan DuckDB: `output/tables/duckdb_summarize_cleaned.csv`

Jalankan `01_data_profiling.ipynb` terlebih dahulu. File mentah tidak diubah.


## Aturan cleaning

1. Mengeluarkan baris `text` bernilai null.
2. Mengeluarkan baris yang kosong atau hanya berisi whitespace.
3. Mempertahankan isi teks nonkosong apa adanya. `strip_chars()` hanya dipakai untuk mendeteksi baris kosong, bukan mengubah konten.
4. Deduplikasi persis tambahan hanya dilakukan jika profiling menemukan duplikasi di luar nilai null/kosong.
5. Memvalidasi hasil dan memperbarui laporan kualitas data.

Teks panjang tidak otomatis dibuang hanya karena jumlah karakternya besar.


In [3]:
from pathlib import Path
import json
import polars as pl
import duckdb

CWD = Path.cwd().resolve()
ROOT = next((p for p in [CWD, *CWD.parents]
             if (p / "data/raw/corpus-indonesia/data").is_dir()), CWD)
RAW_DIR = ROOT / "data" / "raw" / "corpus-indonesia" / "data"
PROCESSED_DIR = ROOT / "data" / "processed"
OUT_DIR = ROOT / "output" / "tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = sorted(RAW_DIR.glob("*.parquet"))
if not FILES:
    raise FileNotFoundError(f"Tidak ditemukan file Parquet di {RAW_DIR}.")

CLEAN_PATH = PROCESSED_DIR / "corpus_indonesia_cleaned.parquet"
REPORT_PATH = OUT_DIR / "data_quality_report.json"
lf = pl.scan_parquet([str(p) for p in FILES])

print("Root proyek:", ROOT)
print("Jumlah file sumber:", len(FILES))
print("Output:", CLEAN_PATH)
print("Polars:", pl.__version__, "| DuckDB:", duckdb.__version__)


Root proyek: C:\Users\raihan bhakti\Desktop\tugas1-corpus-indonesia
Jumlah file sumber: 7
Output: C:\Users\raihan bhakti\Desktop\tugas1-corpus-indonesia\data\processed\corpus_indonesia_cleaned.parquet
Polars: 2.0.0 | DuckDB: 1.5.6


In [4]:
import os
os.environ["POLARS_MAX_THREADS"] = "4"

In [5]:
# Hitung metrik awal dan tentukan apakah deduplikasi tambahan diperlukan.
blank_expr = pl.col("text").is_not_null() & (pl.col("text").str.strip_chars() == "")
pre = lf.select(
    pl.len().alias("total_baris"),
    pl.col("text").null_count().alias("text_null"),
    blank_expr.sum().alias("text_kosong_atau_spasi"),
    pl.col("text").n_unique().alias("jumlah_nilai_unik"),
).collect()
display(pre)
m = pre.row(0, named=True)

rows_before = int(m["total_baris"])
null_count = int(m["text_null"])
blank_count = int(m["text_kosong_atau_spasi"])
unique_values = int(m["jumlah_nilai_unik"])
duplicate_excess = rows_before - unique_values

blank_unique = int(
    lf.filter(blank_expr).select(pl.col("text").n_unique()).collect().item()
) if blank_count else 0
blank_duplicate_excess = max(blank_count - blank_unique, 0)
null_duplicate_excess = max(null_count - 1, 0) if null_count else 0
other_duplicate_excess = duplicate_excess - blank_duplicate_excess - null_duplicate_excess

if other_duplicate_excess < 0:
    raise ValueError("Statistik duplikasi tidak konsisten. Periksa kembali sumber data.")

print(f"Baris sebelum cleaning: {rows_before:,}")
print(f"Null: {null_count:,}")
print(f"Kosong/spasi: {blank_count:,}")
print(f"Duplikasi berlebih total: {duplicate_excess:,}")
print(f"Duplikasi berlebih di luar null/kosong: {other_duplicate_excess:,}")


total_baris,text_null,text_kosong_atau_spasi,jumlah_nilai_unik
u32,u32,u32,u32
19483721,0,1000001,18483721


Baris sebelum cleaning: 19,483,721
Null: 0
Kosong/spasi: 1,000,001
Duplikasi berlebih total: 1,000,000
Duplikasi berlebih di luar null/kosong: 0


In [3]:
# Bersihkan dengan LazyFrame supaya tidak memuat seluruh dataset ke memori.
clean_lf = lf.filter(
    pl.col("text").is_not_null()
    & (pl.col("text").str.strip_chars() != "")
)

if other_duplicate_excess > 0:
    clean_lf = clean_lf.unique(
        subset=["text"],
        keep="first",
        maintain_order=False,
    )
    dedupe_action = (
        "Setelah null/kosong dikeluarkan, deduplikasi persis dilakukan pada kolom text "
        "karena masih ada duplikasi tambahan."
    )
else:
    dedupe_action = (
        "Deduplikasi global tambahan tidak diperlukan berdasarkan profiling: "
        "kelebihan duplikasi dijelaskan oleh nilai null/kosong yang dikeluarkan."
    )

# Jika notebook dijalankan ulang, file output akan dibuat ulang.
if CLEAN_PATH.exists():
    CLEAN_PATH.unlink()

print("Menulis Parquet bersih. Proses dapat memerlukan waktu.")
clean_lf.sink_parquet(str(CLEAN_PATH), compression="zstd")
print("Selesai:", CLEAN_PATH)
print(f"Ukuran hasil: {CLEAN_PATH.stat().st_size / 1_000_000:.2f} MB")
print("Aturan deduplikasi:", dedupe_action)


Menulis Parquet bersih. Proses dapat memerlukan waktu.
Selesai: C:\Users\raihan bhakti\Desktop\tugas1-corpus-indonesia\data\processed\corpus_indonesia_cleaned.parquet
Ukuran hasil: 1195.85 MB
Aturan deduplikasi: Deduplikasi global tambahan tidak diperlukan berdasarkan profiling: kelebihan duplikasi dijelaskan oleh nilai null/kosong yang dikeluarkan.


In [4]:
# Validasi ulang hasil cleaning
cleaned = pl.scan_parquet(str(CLEAN_PATH))
post_blank = pl.col("text").is_not_null() & (pl.col("text").str.strip_chars() == "")
post = cleaned.select(
    pl.len().alias("total_baris_hasil"),
    pl.col("text").null_count().alias("text_null_hasil"),
    post_blank.sum().alias("text_kosong_atau_spasi_hasil"),
    pl.col("text").n_unique().alias("jumlah_nilai_unik_hasil"),
    pl.col("text").str.len_chars().min().alias("panjang_min_karakter"),
    pl.col("text").str.len_chars().max().alias("panjang_maks_karakter"),
    pl.col("text").str.len_chars().mean().alias("panjang_rata_rata"),
).collect()
display(post)
p = post.row(0, named=True)

rows_after = int(p["total_baris_hasil"])
duplicates_after = rows_after - int(p["jumlah_nilai_unik_hasil"])
expected_rows = rows_before - null_count - blank_count - other_duplicate_excess

print(f"Perkiraan baris hasil: {expected_rows:,}")
print(f"Baris aktual hasil: {rows_after:,}")
print(f"Duplikasi berlebih hasil: {duplicates_after:,}")

assert int(p["text_null_hasil"]) == 0, "Masih ada nilai null."
assert int(p["text_kosong_atau_spasi_hasil"]) == 0, "Masih ada teks kosong/spasi."
assert duplicates_after == 0, "Masih ada duplikasi persis."
assert rows_after == expected_rows, "Jumlah baris hasil tidak sesuai aturan cleaning."
print("Validasi berhasil.")


total_baris_hasil,text_null_hasil,text_kosong_atau_spasi_hasil,jumlah_nilai_unik_hasil,panjang_min_karakter,panjang_maks_karakter,panjang_rata_rata
u32,u32,u32,u32,u32,u32,f64
18483720,0,0,18483720,1,180383,170.083601


Perkiraan baris hasil: 18,483,720
Baris aktual hasil: 18,483,720
Duplikasi berlebih hasil: 0
Validasi berhasil.


In [5]:
# Ringkasan hasil bersih menggunakan DuckDB
clean_glob = CLEAN_PATH.as_posix().replace("'", "''")
con = duckdb.connect()
duck_summary = con.execute(f'''
    SELECT COUNT(*) AS total_baris_hasil,
           COUNT(*) FILTER (WHERE text IS NULL) AS text_null_hasil,
           COUNT(*) FILTER (WHERE text IS NOT NULL AND length(trim(text)) = 0) AS text_kosong_atau_spasi_hasil,
           MIN(length(text)) AS panjang_min_karakter,
           MAX(length(text)) AS panjang_maks_karakter,
           AVG(length(text)) AS panjang_rata_rata
    FROM read_parquet('{clean_glob}')
''').df()
display(duck_summary)
duck_csv = OUT_DIR / "duckdb_summarize_cleaned.csv"
duck_summary.to_csv(duck_csv, index=False)
con.close()
print("Ringkasan DuckDB:", duck_csv)


,total_baris_hasil,text_null_hasil,text_kosong_atau_spasi_hasil,panjang_min_karakter,panjang_maks_karakter,panjang_rata_rata
0,18483720,0,0,1,180383,170.083601


Ringkasan DuckDB: C:\Users\raihan bhakti\Desktop\tugas1-corpus-indonesia\output\tables\duckdb_summarize_cleaned.csv


In [6]:
# Perbarui laporan kualitas data
if REPORT_PATH.exists():
    with REPORT_PATH.open("r", encoding="utf-8") as f:
        report = json.load(f)
else:
    report = {
        "dataset": "Corpus-Indonesia",
        "source_url": "https://huggingface.co/datasets/Lyon28/Corpus-Indonesia",
        "source_directory": "data/raw/corpus-indonesia/data/",
        "source_file_count": len(FILES),
        "source_size_bytes": sum(f.stat().st_size for f in FILES),
        "polars_version": pl.__version__,
        "duckdb_version": duckdb.__version__,
    }

def safe(value):
    return value.item() if hasattr(value, "item") else value

report["cleaning_rules"] = [
    "Hapus baris text bernilai null.",
    "Hapus baris text kosong atau hanya whitespace.",
    "Pertahankan isi teks nonkosong apa adanya.",
    dedupe_action,
    "Validasi ulang null, teks kosong, dan duplikasi setelah cleaning.",
]
report["before_cleaning"] = {
    "total_baris": rows_before,
    "text_null": null_count,
    "text_kosong_atau_spasi": blank_count,
    "jumlah_nilai_unik": unique_values,
    "duplikasi_berlebih_total": duplicate_excess,
    "duplikasi_berlebih_di_luar_null_kosong": other_duplicate_excess,
}
report["after_cleaning"] = {key: safe(value) for key, value in p.items()}
report["rows_removed_total"] = rows_before - rows_after
report["rows_removed_null"] = null_count
report["rows_removed_blank_or_whitespace"] = blank_count
report["rows_removed_additional_exact_duplicates"] = other_duplicate_excess
report["rows_after_cleaning"] = rows_after
report["cleaned_file"] = "data/processed/corpus_indonesia_cleaned.parquet"
report["cleaned_file_size_bytes"] = CLEAN_PATH.stat().st_size
report["duckdb_summarize_cleaned_file"] = "output/tables/duckdb_summarize_cleaned.csv"

with REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print("Laporan diperbarui:", REPORT_PATH)
print("Parquet bersih:", CLEAN_PATH)
print("Ringkasan DuckDB:", duck_csv)


Laporan diperbarui: C:\Users\raihan bhakti\Desktop\tugas1-corpus-indonesia\output\tables\data_quality_report.json
Parquet bersih: C:\Users\raihan bhakti\Desktop\tugas1-corpus-indonesia\data\processed\corpus_indonesia_cleaned.parquet
Ringkasan DuckDB: C:\Users\raihan bhakti\Desktop\tugas1-corpus-indonesia\output\tables\duckdb_summarize_cleaned.csv


## Kesimpulan Milestone 2

Gunakan angka aktual dari tabel validasi untuk menjelaskan jumlah baris yang dipertahankan, baris yang dihapus, dan hasil pemeriksaan ulang.

**GitHub:** jangan commit dataset mentah atau Parquet hasil cleaning berukuran besar kecuali dosen mewajibkannya. Pertimbangkan aturan berikut di `.gitignore`:

```gitignore
/data/raw/
/data/processed/*.parquet
```
